# PyTorch Day 2 — Training Concepts Workbook

This is an **unsolved practice notebook** for 2026-07-15. It uses ten focused exercises to build one complete mental model of PyTorch training.

Each section explains:

- What the abstraction means
- Relevant tensor shapes
- What PyTorch does internally
- Common failure modes
- When the API is useful

Replace every placeholder marked YOUR CODE. Read each assertion as a specification, predict the result, and then run the cell. There is intentionally no solutions section.

In [ ]:
import math
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

torch.manual_seed(42)
print("PyTorch:", torch.__version__)

## 1. Write small, testable functions

### Mental model

A function is a named transformation with a contract:

- Input contract: expected shapes, dtypes, and meaning of each dimension.
- Output contract: returned shape and meaning.
- Behavioral contract: whether it mutates inputs, uses randomness, or tracks gradients.

For tensor code, documenting shapes is often more useful than a long description.

~~~python
def add_channel_bias(
    x: torch.Tensor,      # (batch, sequence, channels)
    bias: torch.Tensor,   # (channels,)
) -> torch.Tensor:       # (batch, sequence, channels)
    return x + bias       # broadcasts over batch and sequence
~~~

Small functions are easy to test with tiny inputs. Avoid reading unrelated global variables; pass dependencies as arguments.

### Why keepdim matters in normalization

For x shaped (examples, features), reducing dim=0 calculates one statistic per feature:

~~~python
mean = x.mean(dim=0, keepdim=True)  # (1, features)
std = x.std(dim=0, keepdim=True, unbiased=False)
~~~

keepdim=True retains a size-1 axis, so subtraction and division broadcast back across all examples. In production code, add a small epsilon before division so a constant feature does not cause division by zero.

In [ ]:
# EXERCISE 1 — standardize each feature/column
# Input: (n_examples, n_features). Output must have the same shape.
# Each output column should have mean 0 and population std 1.

def standardize_features(x: torch.Tensor) -> torch.Tensor:
    ...  # YOUR CODE

x = torch.randn(32, 5) * torch.tensor([1., 2., 3., 4., 5.]) + 10
z = standardize_features(x)

assert z.shape == x.shape
assert torch.allclose(z.mean(dim=0), torch.zeros(5), atol=1e-6)
assert torch.allclose(z.std(dim=0, unbiased=False), torch.ones(5), atol=1e-5)
z[:3]

## 2. Autograd

### What PyTorch records

When a tensor has requires_grad=True, PyTorch builds a dynamic computation graph as operations run. Each result remembers the operation that created it and the earlier tensors it depends on.

~~~python
x = torch.tensor(3.0, requires_grad=True)  # leaf tensor
y = x * x + 4
y.backward()                              # chain rule from y back to x
print(x.grad)                             # dy/dx evaluated at x=3
~~~

A leaf tensor is normally a value created directly, such as a model parameter. Its gradient is stored in .grad.

### Why training losses are scalars

backward() needs a starting derivative. For a scalar loss, that derivative is unambiguously 1. If an output has many elements, reduce it with mean() or sum(), or explicitly supply a gradient.

Three details prevent many bugs:

- Gradients accumulate. A second backward call adds to the existing .grad.
- detach() returns the same value disconnected from the graph.
- torch.no_grad() prevents graph construction and saves memory during evaluation.

After backward(), inspect whether parameter gradients are None and whether they contain finite values.

In [ ]:
# EXERCISE 2
# At x=2, calculate y = 3x^2 + 2x + 1 and populate x.grad.
x = torch.tensor(2.0, requires_grad=True)
y = ...  # YOUR CODE
...  # YOUR CODE: run backpropagation

assert y.item() == 17.0
assert x.grad is not None
assert x.grad.item() == 14.0
x.grad

## 3. Linear models using tensor operations

A linear layer calculates weighted combinations of input features:

~~~text
x:       (batch, input_features)
weight:  (input_features, output_features)
bias:    (output_features,)
output:  (batch, output_features)
~~~

The manual operation is x @ weight + bias. Matrix multiplication contracts the shared input_features dimension. The bias broadcasts across the batch.

### Important nn.Linear detail

nn.Linear(in_features, out_features) stores its weight as (out_features, in_features), the reverse of the manual weight above. Its computation is conceptually:

~~~python
output = x @ layer.weight.T + layer.bias
~~~

Remember this when copying weights or comparing manual code with nn.Linear.

A stack of linear layers without activations is still equivalent to a single linear layer. ReLU or another activation is what allows the network to represent nonlinear relationships.

In [ ]:
# EXERCISE 3 — implement a linear forward pass without nn.Linear
def linear_forward(
    x: torch.Tensor,
    weight: torch.Tensor,
    bias: torch.Tensor,
) -> torch.Tensor:
    ...  # YOUR CODE

x = torch.randn(8, 4)
weight = torch.randn(4, 3)
bias = torch.randn(3)
output = linear_forward(x, weight, bias)

assert output.shape == (8, 3)
assert torch.allclose(output[0], x[0] @ weight + bias)
output.shape

## 4. Loss functions

A loss converts predictions and targets into one scalar: how wrong is this batch? Autograd then calculates how every parameter influenced it.

For regression, mean squared error compares continuous values. Prediction and target normally have identical shapes.

~~~python
loss = ((prediction - target) ** 2).mean()
loss = nn.functional.mse_loss(prediction, target)
~~~

### Classification uses logits

Logits are unrestricted real-valued class scores, not probabilities.

~~~text
logits: (batch, number_of_classes), floating point
labels: (batch,), integer class indices using torch.int64
~~~

~~~python
loss = nn.functional.cross_entropy(logits, labels)
~~~

Cross-entropy already performs a numerically stable log-softmax internally. Do not apply softmax first.

Common errors are float labels, one-hot labels where indices are expected, logits shaped (batch,) instead of (batch, classes), and applying softmax before the loss.

In [ ]:
# EXERCISE 4 — classification loss
# Calculate cross-entropy directly from the raw logits.
logits = torch.tensor([[3.0, 1.0, -1.0], [0.0, 2.0, 1.0]])
labels = torch.tensor([0, 2])
loss = ...  # YOUR CODE

assert loss.ndim == 0
assert 0 < loss.item() < 2
loss

## 5. Optimizers and one training step

An optimizer holds references to parameters and updates them using their .grad tensors. Basic SGD is conceptually:

~~~text
parameter = parameter - learning_rate * gradient
~~~

The order of a training step matters:

~~~python
optimizer.zero_grad()              # clear gradients from the previous batch
prediction = model(features)       # forward pass builds a graph
loss = loss_fn(prediction, target) # scalar error
loss.backward()                    # populate parameter.grad
optimizer.step()                   # change parameter values
~~~

zero_grad clears gradients, not weights. backward calculates gradients but does not update weights. step updates weights but does not calculate gradients.

The learning rate controls update size. Too small learns slowly; too large can oscillate, explode, or create NaNs. Comparing a parameter before and after step() confirms that training is actually changing the model.

In [ ]:
# EXERCISE 5 — perform exactly one optimization step
model = nn.Linear(1, 1)
optimizer = torch.optim.SGD(model.parameters(), lr=0.1)
x = torch.tensor([[1.0], [2.0], [3.0]])
target = 2 * x + 1

weight_before = model.weight.detach().clone()

...  # YOUR CODE: clear gradients
prediction = ...  # YOUR CODE
loss = ...  # YOUR CODE: MSE
...  # YOUR CODE: backward
...  # YOUR CODE: update parameters

assert loss.ndim == 0
assert model.weight.grad is not None
assert not torch.equal(model.weight.detach(), weight_before)
loss.item()

## 6. Building an nn.Module

An nn.Module is a container for parameters and computation.

~~~python
class SmallNet(nn.Module):
    def __init__(self, input_size: int, output_size: int):
        super().__init__()
        self.layer = nn.Linear(input_size, output_size)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.layer(x)
~~~

Assigning a layer to self.layer registers its parameters. Registration is why model.parameters(), state_dict(), model.to(device), and optimizers can find them. A plain tensor is not automatically trainable; nn.Parameter registers a tensor directly.

Call model(x), not model.forward(x), because model(x) also runs PyTorch hooks.

For classification, the final layer returns raw logits. Put ReLU between hidden layers, not after the final layer when cross-entropy is the loss.

In [ ]:
# EXERCISE 6 — build a two-layer classifier
# Architecture: input -> Linear -> ReLU -> Linear -> raw logits
class TwoLayerClassifier(nn.Module):
    def __init__(self, input_size: int, hidden_size: int, num_classes: int):
        super().__init__()
        ...  # YOUR CODE: define layers

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        ...  # YOUR CODE

model = TwoLayerClassifier(input_size=4, hidden_size=16, num_classes=3)
x = torch.randn(8, 4)
logits = model(x)

assert logits.shape == (8, 3)
assert sum(p.numel() for p in model.parameters()) == 131
logits.shape

## 7. Datasets and DataLoaders

A Dataset answers: how many examples exist, and what is example i? A DataLoader decides how examples are grouped and ordered during iteration.

~~~python
dataset = TensorDataset(features, labels)
loader = DataLoader(
    dataset,
    batch_size=32,
    shuffle=True,
    drop_last=False,
)
~~~

TensorDataset keeps tensors aligned by their first dimension. If it returns one feature vector shaped (features,), the loader stacks several into (batch, features).

For 100 examples and batch size 32, len(loader) is 4: three full batches and one batch of 4. drop_last=True would discard the last 4.

Use shuffle=True for training so ordering changes each epoch. Validation loaders normally use shuffle=False for repeatability. Shuffling preserves the feature-label pairing.

In [ ]:
# EXERCISE 7
# Build a shuffled loader with batch size 16 from these tensors.
features = torch.randn(100, 4)
labels = torch.randint(0, 3, (100,))
dataset = ...  # YOUR CODE
loader = ...  # YOUR CODE

feature_batch, label_batch = next(iter(loader))
assert len(dataset) == 100
assert len(loader) == math.ceil(100 / 16)
assert feature_batch.shape == (16, 4)
assert label_batch.shape == (16,)
feature_batch.shape, label_batch.shape

## 8. Synthetic data and sanity checks

Synthetic data helps debug because you know the rule that created the labels. If a small model cannot learn a simple synthetic rule, the pipeline probably contains a bug.

This workbook uses hidden linear class scores:

~~~python
features = torch.randn(number_of_examples, 4)  # (n, 4)
true_weight = torch.randn(4, 3)                # one column per class
scores = features @ true_weight                # (n, 3)
labels = scores.argmax(dim=1)                  # (n,), class indices
~~~

argmax(dim=1) chooses the highest-scoring class for each example and returns int64 indices, exactly what cross-entropy expects.

Before training, inspect feature and label shapes, their dtypes, finite values, and torch.bincount(labels). Severe class imbalance can make raw accuracy misleading.

For real work, make the train/validation split before learning preprocessing statistics. Otherwise information can leak from validation data into training.

## 9. Write a reusable training function

A reusable epoch function receives the model, loader, loss function, and optimizer instead of relying on globals.

For every batch it must clear old gradients, run the forward pass, calculate scalar loss, backpropagate, update parameters, and accumulate a reporting value.

Call model.train() first. It enables training behavior in dropout and batch normalization; it does not calculate gradients by itself.

### Correct mean epoch loss

Most losses return a mean within the current batch. Weight it by batch size before accumulating:

~~~python
total_loss += batch_loss.item() * batch_size
total_examples += batch_size
epoch_loss = total_loss / total_examples
~~~

Simply averaging batch means is slightly wrong when the final batch is smaller.

Use .item() for logging only after the tensor loss has been used for backward. A Python float is disconnected from autograd.

In [ ]:
# EXERCISE 9 — complete one training epoch
def train_one_epoch(
    model: nn.Module,
    loader: DataLoader,
    loss_fn: nn.Module,
    optimizer: torch.optim.Optimizer,
) -> float:
    ...  # YOUR CODE

# This test uses a fresh, independent dataset.
torch.manual_seed(10)
train_x = torch.randn(128, 4)
train_y = (train_x @ true_weight).argmax(dim=1)
train_loader = DataLoader(TensorDataset(train_x, train_y), batch_size=32, shuffle=True)
test_model = TwoLayerClassifier(4, 16, 3)
test_optimizer = torch.optim.SGD(test_model.parameters(), lr=0.1)
average_loss = train_one_epoch(test_model, train_loader, nn.CrossEntropyLoss(), test_optimizer)

assert isinstance(average_loss, float)
assert average_loss > 0
assert all(parameter.grad is not None for parameter in test_model.parameters())
average_loss

## 10. Evaluation

Evaluation measures a fixed model and must not update it.

~~~python
model.eval()
with torch.no_grad():
    logits = model(features)
    predictions = logits.argmax(dim=1)
~~~

model.eval() changes dropout and batch-normalization behavior. no_grad() stops graph construction and saves memory. They solve different problems, so use both.

Softmax is unnecessary for choosing a class because it preserves ordering: argmax of logits and argmax of probabilities are identical.

Accumulate counts across all batches:

~~~python
correct += (predictions == labels).sum().item()
total += labels.numel()
accuracy = correct / total
~~~

This handles a smaller final batch correctly. Averaging unweighted batch accuracies does not.

In [ ]:
# EXERCISE 10 — implement accuracy over a loader
def accuracy(model: nn.Module, loader: DataLoader) -> float:
    ...  # YOUR CODE

# A model with fixed logits for a predictable test.
class IdentityLogitModel(nn.Module):
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return x

test_logits = torch.tensor([
    [5.0, 0.0, 0.0],
    [0.0, 5.0, 0.0],
    [0.0, 0.0, 5.0],
    [5.0, 0.0, 0.0],
])
test_labels = torch.tensor([0, 1, 1, 0])  # three of four are correct
test_loader = DataLoader(TensorDataset(test_logits, test_labels), batch_size=3)
score = accuracy(IdentityLogitModel(), test_loader)

assert isinstance(score, float)
assert score == 0.75
score

## 11. Device-safe code

All tensors participating in an operation must be on compatible devices. A CPU input cannot be multiplied by a CUDA or MPS weight.

~~~python
if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

model = model.to(device)
features = features.to(device)
labels = labels.to(device)
~~~

Important details:

- tensor.to(device) returns a tensor; assign the result.
- Move labels as well as features because the loss uses both.
- Prefer creating the optimizer after moving the model.
- Convert logged scalar tensors with .item(); move larger results to .cpu() before NumPy.
- Debug on CPU first. An accelerator makes correct code faster, not incorrect code clearer.

Treat the device as a value passed into code, not a hard-coded string scattered through functions.

# Final challenge — train the classifier

Connect the complete pipeline:

~~~text
features + labels
       ↓
TensorDataset
       ↓
DataLoader
       ↓
nn.Module → raw logits
       ↓
CrossEntropyLoss
       ↓
backward → optimizer.step
       ↓
evaluation accuracy
~~~

The target is at least 90% training accuracy. That confirms the mechanics work; it does not prove generalization because the same data is used for training and measurement.

Debug in this order:

1. Data: features are (batch, 4); labels are (batch,) int64.
2. Forward: logits are finite and shaped (batch, 3).
3. Loss: scalar, finite, and requires_grad is True.
4. Gradients: parameters have non-None, finite gradients after backward.
5. Updates: at least one parameter changes after step.
6. Learning: epoch loss trends downward, though individual batches can fluctuate.

A strong diagnostic is to overfit 16–32 examples. If a model cannot nearly memorize a tiny subset, fix the pipeline before tuning architecture.

In [ ]:
# FINAL CHALLENGE
# Create the dataset, loader, model, loss, and optimizer. Train for up to 50 epochs.
# Print the loss every 10 epochs and stop early if accuracy reaches 95%.

torch.manual_seed(7)
features = torch.randn(600, 4)
true_weight = torch.tensor([
    [2.0, -1.0, 0.5],
    [-1.0, 2.0, 0.5],
    [0.5, -1.0, 2.0],
    [1.0, 1.0, -1.0],
])
labels = (features @ true_weight).argmax(dim=1)

dataset = ...  # YOUR CODE
loader = ...  # YOUR CODE
model = ...  # YOUR CODE
loss_fn = ...  # YOUR CODE
optimizer = ...  # YOUR CODE

history = []
for epoch in range(50):
    ...  # YOUR CODE

final_accuracy = accuracy(model, loader)
assert len(history) >= 1
assert history[-1] < history[0]
assert final_accuracy >= 0.90, f"Only reached {final_accuracy:.1%}"
final_accuracy

## Optional extension

Once the final challenge passes, choose one modification and predict what will happen before running it:

- Change SGD to Adam.
- Remove the hidden layer and use one `nn.Linear(4, 3)`.
- Try learning rates `0.001`, `0.01`, `0.1`, and `1.0`.
- Add a validation split and report training versus validation accuracy.
- Intentionally forget `zero_grad()` and inspect gradient sizes.
- Save `model.state_dict()`, create a fresh model, and reload it.